In [1]:
from pathlib import Path
import os

# Configure antes de inicializar CUDA.
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("MPLBACKEND", "Agg")

BASE_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
# None = busca automática. Para escolher um dataset, use Path("caminho_real").
DATA_ROOT = None
OUTPUT_BASE = BASE_DIR / "parte3_eixo2"
SOURCE_ROOT = BASE_DIR / "pa2_parte3_modulos"

PUBLIC_DETECTOR = "FRCNN"
WINDOW = 16
STRIDE = 4
SEEDS = (0, 1, 2)
EPOCHS = 20
BATCH_SIZE = 256
HIDDEN_SIZE = 32
LEARNING_RATE = 1e-3
SMOOTH_L1_BETA = 0.01
CLIP_NORM = 1.0
ASSOCIATION_IOU = 0.3
MAX_MISSED = 5

SMOKE_TEST = False
SMOKE_MAX_WINDOWS = 32
SMOKE_MAX_FRAMES = 30
OUTPUT = OUTPUT_BASE / ("smoke_test" if SMOKE_TEST else "experimento_completo")
OUTPUT.mkdir(parents=True, exist_ok=True)

print("Modo:", "SMOKE TEST — apenas verificação" if SMOKE_TEST else "EXPERIMENTO COMPLETO")
print("Resultados:", OUTPUT)


Modo: EXPERIMENTO COMPLETO
Resultados: /kaggle/working/parte3_eixo2/experimento_completo


In [2]:
import csv
import hashlib
import importlib
import json
import sys
import html

import numpy as np
import scipy
import torch
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import HTML, Image, FileLink, display

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
PACKAGE_DIR = SOURCE_ROOT / "pa2"
PACKAGE_DIR.mkdir(parents=True, exist_ok=True)
(PACKAGE_DIR / "__init__.py").write_text('# Módulos reutilizáveis da Parte 3.\n', encoding="utf-8")

print("PyTorch:", torch.__version__, "| NumPy:", np.__version__, "| SciPy:", scipy.__version__)
print("Dispositivo:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
print("Módulos reutilizáveis:", PACKAGE_DIR)


PyTorch: 2.11.0+cu128 | NumPy: 2.1.3 | SciPy: 1.16.3
Dispositivo: cuda
GPU: Tesla T4
Módulos reutilizáveis: /kaggle/working/pa2_parte3_modulos/pa2


In [3]:
%%writefile {PACKAGE_DIR}/gru_model.py
"""A generic GRU sequence-to-sequence model."""

from __future__ import annotations

import torch
from torch import nn


class GRUSequenceModel(nn.Module):
    def __init__(
        self,
        input_size: int,
        hidden_size: int,
        output_size: int,
        num_layers: int = 1,
        dropout: float = 0.0,
        bidirectional: bool = False,
        batch_first: bool = True,
    ) -> None:
        super().__init__()

        if input_size <= 0:
            raise ValueError("input_size must be positive")
        if hidden_size <= 0:
            raise ValueError("hidden_size must be positive")
        if output_size <= 0:
            raise ValueError("output_size must be positive")
        if num_layers <= 0:
            raise ValueError("num_layers must be positive")
        if not 0.0 <= dropout < 1.0:
            raise ValueError("dropout must be in the range [0, 1)")

        self.input_size = input_size
        self.hidden_size = hidden_size
        self.output_size = output_size
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        self.batch_first = batch_first

        directions = 2 if bidirectional else 1
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=batch_first,
            bidirectional=bidirectional,
        )
        self.dropout = nn.Dropout(dropout)
        self.output_projection = nn.Linear(hidden_size * directions, output_size)

    def forward(
        self,
        sequence: torch.Tensor,
        hidden: torch.Tensor | None = None,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        """Return per-step predictions and the final GRU hidden state."""
        if sequence.ndim != 3:
            raise ValueError(
                "sequence must have three dimensions "
                "(batch, time, features) or (time, batch, features)"
            )
        if sequence.shape[-1] != self.input_size:
            raise ValueError(
                f"Expected {self.input_size} input features, got {sequence.shape[-1]}"
            )

        recurrent_output, final_hidden = self.gru(sequence, hidden)
        predictions = self.output_projection(self.dropout(recurrent_output))
        return predictions, final_hidden


Writing /kaggle/working/pa2_parte3_modulos/pa2/gru_model.py


In [5]:
%%writefile {PACKAGE_DIR}/mot17_dataset.py
"""PyTorch datasets for MOT17 in MOTChallenge directory layout."""

from __future__ import annotations

import configparser
import csv
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset


@dataclass
class _Sequence:
    """Loaded MOT17 sequence metadata and per-frame annotations."""

    name: str
    image_paths: list[Path]
    frame_numbers: list[int]
    boxes: dict[int, np.ndarray]
    ids: dict[int, np.ndarray]


def _sequence_dirs(root: Path, split: str) -> list[Path]:
    images_root = root / "MOT17" / split
    return sorted(p.parent for p in images_root.glob("*/seqinfo.ini"))


def _find_gt(root: Path, split: str, sequence_dir: Path) -> Path | None:
    candidates = (
        sequence_dir / "gt" / "gt.txt",
        root / "MOT17Labels" / split / sequence_dir.name / "gt" / "gt.txt",
    )
    return next((path for path in candidates if path.is_file()), None)


def _read_annotations(path: Path) -> tuple[dict[int, list[list[float]]], dict[int, list[int]]]:
    boxes: dict[int, list[list[float]]] = {}
    ids: dict[int, list[int]] = {}

    with path.open(newline="") as f:
        for row in csv.reader(f):
            if not row:
                continue
            try:
                values = [float(value) for value in row]
            except ValueError as exc:
                raise ValueError(f"Invalid MOT annotation row in {path}: {row}") from exc

            if len(values) < 6:
                raise ValueError(f"Expected at least 6 columns in {path}, got: {row}")

            frame, object_id = int(values[0]), int(values[1])

            # MOT17 rows are frame,id,x,y,w,h,mark,class,visibility. Keep
            # marked pedestrian annotations and ignore distractor classes.
            if len(values) >= 8 and (values[6] != 1 or values[7] != 1):
                continue

            boxes.setdefault(frame, []).append(values[2:6])
            ids.setdefault(frame, []).append(object_id)

    return boxes, ids


def _load_sequences(root: Path, split: str) -> list[_Sequence]:
    if not root.is_dir():
        raise FileNotFoundError(
            f"MOT17 data directory not found: {root}. "
            "Place the dataset under data/MOT17 (and optionally data/MOT17Labels)."
        )

    dirs = _sequence_dirs(root, split)
    if not dirs:
        raise FileNotFoundError(
            f"No MOT17 sequences found under {root / 'MOT17' / split}. "
            "Expected folders containing seqinfo.ini and an image directory."
        )

    sequences = []
    missing_gt = []
    for sequence_dir in dirs:
        info_path = sequence_dir / "seqinfo.ini"
        config = configparser.ConfigParser()
        config.read(info_path)
        if not config.has_section("Sequence"):
            raise ValueError(f"Missing [Sequence] section in {info_path}")

        info = config["Sequence"]
        image_dir = sequence_dir / info.get("imDir", "img1")
        image_ext = info.get("imExt", ".jpg")
        if not image_ext.startswith("."):
            image_ext = f".{image_ext}"

        # Use the numbered image files actually present rather than assuming
        # every sequence has all frames (useful for partial local datasets).
        image_by_frame = {}
        for image_path in image_dir.glob(f"*{image_ext}"):
            try:
                image_by_frame[int(image_path.stem)] = image_path
            except ValueError:
                continue
        frame_numbers = sorted(image_by_frame)
        if not frame_numbers:
            continue

        gt_path = _find_gt(root, split, sequence_dir)
        if gt_path is None:
            missing_gt.append(sequence_dir.name)
            continue

        raw_boxes, raw_ids = _read_annotations(gt_path)
        sequences.append(
            _Sequence(
                name=sequence_dir.name,
                image_paths=[image_by_frame[frame] for frame in frame_numbers],
                frame_numbers=frame_numbers,
                boxes={
                    frame: np.asarray(raw_boxes.get(frame, []), dtype=np.float32).reshape(-1, 4)
                    for frame in frame_numbers
                },
                ids={
                    frame: np.asarray(raw_ids.get(frame, []), dtype=np.int32)
                    for frame in frame_numbers
                },
            )
        )

    if not sequences:
        details = f" Sequences without ground-truth labels: {', '.join(missing_gt)}." if missing_gt else ""
        raise FileNotFoundError(
            f"No usable labeled MOT17 sequences found for split '{split}' under {root}."
            f"{details} Expected labels at <sequence>/gt/gt.txt or "
            "MOT17Labels/<split>/<sequence>/gt/gt.txt."
        )

    return sequences


class MOT17FrameDataset(Dataset):
    def __init__(self, root="data", split="train"):
        self.sequences = _load_sequences(Path(root), split)
        self.index = [
            (sequence_index, image_index)
            for sequence_index, sequence in enumerate(self.sequences)
            for image_index in range(len(sequence.image_paths))
        ]

    def __len__(self):
        return len(self.index)

    def __getitem__(self, index):
        sequence_index, image_index = self.index[index]
        sequence = self.sequences[sequence_index]
        frame = sequence.frame_numbers[image_index]

        with Image.open(sequence.image_paths[image_index]) as image_file:
            image = np.asarray(image_file.convert("RGB"))
        image_tensor = torch.from_numpy(image.copy()).permute(2, 0, 1).float() / 255.0
        boxes = torch.from_numpy(sequence.boxes[frame]).float()
        ids = torch.from_numpy(sequence.ids[frame]).int()
        return image_tensor, boxes, ids


class MOT17TrackDataset(Dataset):
    def __init__(self, root="data", split="train", T=16, stride=4):
        if T <= 0:
            raise ValueError("T must be a positive integer")
        if stride <= 0:
            raise ValueError("stride must be a positive integer")

        self.T = T
        self.samples = []
        self.sample_sequences = []
        for sequence in _load_sequences(Path(root), split):
            frame_to_index = {frame: index for index, frame in enumerate(sequence.frame_numbers)}
            all_ids = sorted({int(object_id) for values in sequence.ids.values() for object_id in values})

            for object_id in all_ids:
                track = [None] * len(sequence.frame_numbers)
                for frame in sequence.frame_numbers:
                    matching = np.flatnonzero(sequence.ids[frame] == object_id)
                    if len(matching):
                        track[frame_to_index[frame]] = sequence.boxes[frame][matching[0]]

                for start in range(0, len(track) - T + 1, stride):
                    box_seq = np.zeros((T, 4), dtype=np.float32)
                    valid = np.zeros((T,), dtype=np.float32)
                    for offset, box in enumerate(track[start : start + T]):
                        if box is not None:
                            box_seq[offset] = box
                            valid[offset] = 1.0
                    if valid.sum() >= 2:
                        self.samples.append((box_seq, valid))
                        self.sample_sequences.append(sequence.name)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        box_seq, valid = self.samples[index]
        return torch.from_numpy(box_seq), torch.from_numpy(valid)


Overwriting /kaggle/working/pa2_parte3_modulos/pa2/mot17_dataset.py


In [6]:
%%writefile {PACKAGE_DIR}/datasets.py

"""Deterministic, sequence-level MOT17 train/validation/test splits."""

from torch.utils.data import Subset

from pa2.mot17_dataset import MOT17FrameDataset, MOT17TrackDataset


# Split by source sequence number, so DPM/FRCNN/SDP variants stay together.
# MOT17-05 is the low-resolution (640x480), low-frame-rate outlier and is
# reserved for validation. MOT17-13 is held out for final testing.
MOT17_SEQUENCE_SPLITS = {
    "train": ("02", "04", "09", "10", "11"),
    "validation": ("05",),
    "test": ("13",),
}


def _sequence_number(name: str) -> str:
    """Extract the source sequence ID from names such as MOT17-02-SDP."""
    parts = name.split("-")
    if len(parts) < 3 or parts[0] != "MOT17":
        raise ValueError(f"Unexpected MOT17 sequence name: {name}")
    return parts[1]


def _split_for_sequence(name: str) -> str:
    number = _sequence_number(name)
    for split, sequence_numbers in MOT17_SEQUENCE_SPLITS.items():
        if number in sequence_numbers:
            return split
    raise ValueError(f"No dataset split configured for MOT17 sequence {name}")


def make_mot17_frame_splits(root="data") -> dict[str, Subset[MOT17FrameDataset]]:
    """Return frame datasets split by whole source sequence, never by frame."""
    dataset = MOT17FrameDataset(root=root, split="train")
    split_indices = {split: [] for split in MOT17_SEQUENCE_SPLITS}

    for index, (sequence_index, _) in enumerate(dataset.index):
        sequence = dataset.sequences[sequence_index]
        split_indices[_split_for_sequence(sequence.name)].append(index)

    return {split: Subset(dataset, indices) for split, indices in split_indices.items()}


def make_mot17_track_splits(root="data", T=16, stride=4) -> dict[str, Subset[MOT17TrackDataset]]:
    """Return track windows split by source sequence, never by frame."""
    dataset = MOT17TrackDataset(root=root, split="train", T=T, stride=stride)
    split_indices = {split: [] for split in MOT17_SEQUENCE_SPLITS}
    for index, sequence_name in enumerate(dataset.sample_sequences):
        split_indices[_split_for_sequence(sequence_name)].append(index)
    return {split: Subset(dataset, indices) for split, indices in split_indices.items()}


Writing /kaggle/working/pa2_parte3_modulos/pa2/datasets.py


In [8]:
%%writefile {PACKAGE_DIR}/metrics.py
"""Own association and trajectory metrics. All boxes use MOT (x, y, w, h)."""

from __future__ import annotations

import numpy as np
from scipy.optimize import linear_sum_assignment

Box = tuple[float, float, float, float]


def iou_matrix(boxes_a, boxes_b) -> np.ndarray:
    a = np.asarray(boxes_a, dtype=float).reshape(-1, 4)
    b = np.asarray(boxes_b, dtype=float).reshape(-1, 4)
    top_left = np.maximum(a[:, None, :2], b[None, :, :2])
    bottom_right = np.minimum(a[:, None, :2] + a[:, None, 2:],
                              b[None, :, :2] + b[None, :, 2:])
    intersection = np.maximum(bottom_right - top_left, 0).prod(axis=-1)
    area_a = np.maximum(a[:, 2:], 0).prod(axis=1)
    area_b = np.maximum(b[:, 2:], 0).prod(axis=1)
    union = area_a[:, None] + area_b[None, :] - intersection
    return np.divide(intersection, union, out=np.zeros_like(intersection), where=union > 0)


def box_iou(box1: Box, box2: Box) -> float:
    return float(iou_matrix([box1], [box2])[0, 0])


def gated_hungarian(similarities: np.ndarray, threshold: float) -> list[tuple[int, int]]:
    """Maximize valid-pair count first, then IoU. Gate BEFORE solving."""
    if not 0 < threshold <= 1:
        raise ValueError("IoU threshold must be in (0, 1]")
    if 0 in similarities.shape:
        return []
    valid = np.isfinite(similarities) & (similarities >= threshold)
    bonus = min(similarities.shape) + 1
    reward = np.where(valid, bonus + similarities, 0.0)
    rows, cols = linear_sum_assignment(-reward)
    return [(int(r), int(c)) for r, c in zip(rows, cols) if valid[r, c]]


def match_frame(gt_boxes, gt_ids, pred_boxes, pred_ids, iou_thresh=0.5) -> list[tuple[int, int]]:
    pairs = gated_hungarian(iou_matrix(gt_boxes, pred_boxes), iou_thresh)
    return [(int(gt_ids[r]), int(pred_ids[c])) for r, c in pairs]


def _validate_frames(gt_boxes, gt_ids, pred_boxes, pred_ids):
    if len({len(gt_boxes), len(gt_ids), len(pred_boxes), len(pred_ids)}) != 1:
        raise ValueError("GT and predictions must contain the same evaluated frames")
    for boxes, ids in zip(list(gt_boxes) + list(pred_boxes), list(gt_ids) + list(pred_ids)):
        if len(boxes) != len(ids) or len(set(map(int, ids))) != len(ids):
            raise ValueError("Every box needs one identity, unique within its frame")


def compute_idf1(gt_boxes, gt_ids, pred_boxes, pred_ids, iou_thresh=0.5):
    _validate_frames(gt_boxes, gt_ids, pred_boxes, pred_ids)
    gt_index = {x: i for i, x in enumerate(sorted({int(x) for f in gt_ids for x in f}))}
    pred_index = {x: i for i, x in enumerate(sorted({int(x) for f in pred_ids for x in f}))}
    overlap = np.zeros((len(gt_index), len(pred_index)), dtype=np.int64)
    for g_boxes, g_ids, p_boxes, p_ids in zip(gt_boxes, gt_ids, pred_boxes, pred_ids):
        for g, p in zip(*np.where(iou_matrix(g_boxes, p_boxes) >= iou_thresh)):
            overlap[gt_index[int(g_ids[g])], pred_index[int(p_ids[p])]] += 1
    idtp = 0
    if overlap.size:
        rows, cols = linear_sum_assignment(-overlap)
        idtp = int(overlap[rows, cols].sum())
    idfn = sum(map(len, gt_boxes)) - idtp
    idfp = sum(map(len, pred_boxes)) - idtp
    denominator = 2 * idtp + idfp + idfn
    return (2 * idtp / denominator if denominator else 1.0,
            {"IDTP": idtp, "IDFP": idfp, "IDFN": idfn})


def _identity_events(gt_boxes, gt_ids, pred_boxes, pred_ids, iou_thresh):
    previous, last_id = {}, {}
    seen, interrupted = set(), set()
    events = []
    for t, (g_boxes, g_ids, p_boxes, p_ids) in enumerate(zip(gt_boxes, gt_ids, pred_boxes, pred_ids)):
        similarity = iou_matrix(g_boxes, p_boxes)
        p_index = {int(identity): i for i, identity in enumerate(p_ids)}
        pairs, used_g, used_p = [], set(), set()
        for g, identity in enumerate(g_ids):
            p = p_index.get(previous.get(int(identity)))
            if p is not None and similarity[g, p] >= iou_thresh:
                pairs.append((g, p))
                used_g.add(g)
                used_p.add(p)
        remaining_g = [g for g in range(len(g_ids)) if g not in used_g]
        remaining_p = [p for p in range(len(p_ids)) if p not in used_p]
        pairs.extend((remaining_g[g], remaining_p[p]) for g, p in
                     gated_hungarian(similarity[np.ix_(remaining_g, remaining_p)], iou_thresh))
        current = {int(g_ids[g]): int(p_ids[p]) for g, p in pairs}
        for identity in map(int, g_ids):
            if identity in current:
                prediction = current[identity]
                if identity in last_id and last_id[identity] != prediction:
                    events.append({"frame_index": t, "event": "ID_switch", "gt_id": identity,
                                   "old_pred_id": last_id[identity], "new_pred_id": prediction})
                if identity in interrupted:
                    events.append({"frame_index": t, "event": "fragmentation", "gt_id": identity,
                                   "old_pred_id": last_id[identity], "new_pred_id": prediction})
                    interrupted.remove(identity)
                seen.add(identity)
                last_id[identity] = prediction
            elif identity in seen:
                interrupted.add(identity)
        previous = current
    return events


def count_id_switches(gt_boxes, gt_ids, pred_boxes, pred_ids, iou_thresh=0.5) -> int:
    _validate_frames(gt_boxes, gt_ids, pred_boxes, pred_ids)
    return sum(e["event"] == "ID_switch" for e in
               _identity_events(gt_boxes, gt_ids, pred_boxes, pred_ids, iou_thresh))


def count_fragmentations(gt_boxes, gt_ids, pred_boxes, pred_ids, iou_thresh=0.5) -> int:
    _validate_frames(gt_boxes, gt_ids, pred_boxes, pred_ids)
    return sum(e["event"] == "fragmentation" for e in
               _identity_events(gt_boxes, gt_ids, pred_boxes, pred_ids, iou_thresh))


def frame_ap(gt_boxes, pred_boxes, scores, iou_thresh=0.5) -> float:
    """Confidence-ordered, precision-envelope integral AP for one person frame."""
    if not len(gt_boxes):
        return float("nan")
    similarity = iou_matrix(gt_boxes, pred_boxes)
    used_gt, true_positive = set(), []
    for p in np.argsort(-np.asarray(scores), kind="stable"):
        candidates = [g for g in range(len(gt_boxes)) if g not in used_gt]
        g = max(candidates, key=lambda i: similarity[i, p]) if candidates else None
        matched = g is not None and similarity[g, p] >= iou_thresh
        if matched:
            used_gt.add(g)
        true_positive.append(int(matched))
    if not true_positive:
        return 0.0
    tp = np.cumsum(true_positive)
    recall = np.r_[0.0, tp / len(gt_boxes), 1.0]
    precision = np.r_[0.0, tp / np.arange(1, len(tp) + 1), 0.0]
    precision = np.maximum.accumulate(precision[::-1])[::-1]
    return float(np.sum(np.diff(recall) * precision[1:]))


def evaluate_trajectories(
    gt_boxes, gt_ids, pred_boxes, pred_ids, pred_scores=None,
    ignored_boxes=None, iou_thresh=0.5,
) -> tuple[dict, list[dict]]:
    
    _validate_frames(gt_boxes, gt_ids, pred_boxes, pred_ids)
    n_frames = len(gt_boxes)
    if ignored_boxes is not None and len(ignored_boxes) != n_frames:
        raise ValueError("Ignore boxes must contain every evaluated frame")
    if pred_scores is not None and (len(pred_scores) != n_frames or
                                   any(len(s) != len(b) for s, b in zip(pred_scores, pred_boxes))):
        raise ValueError("Prediction scores must align with boxes")
    filtered_boxes, filtered_ids, filtered_scores = [], [], []
    raw_ids = {int(x) for frame in pred_ids for x in frame}
    for t in range(n_frames):
        boxes = np.asarray(pred_boxes[t], dtype=float).reshape(-1, 4)
        ids = np.asarray(pred_ids[t], dtype=np.int64)
        keep = np.ones(len(boxes), dtype=bool)
        if ignored_boxes is not None and len(ignored_boxes[t]) and len(boxes):
            protected = {p for _, p in gated_hungarian(iou_matrix(gt_boxes[t], boxes), iou_thresh)}
            ignored = iou_matrix(boxes, ignored_boxes[t]).max(axis=1) >= iou_thresh
            keep = np.asarray([not ignored[p] or p in protected for p in range(len(boxes))])
        filtered_boxes.append(boxes[keep])
        filtered_ids.append(ids[keep])
        if pred_scores is not None:
            filtered_scores.append(np.asarray(pred_scores[t])[keep])
    idf1, counts = compute_idf1(gt_boxes, gt_ids, filtered_boxes, filtered_ids, iou_thresh)
    events = _identity_events(gt_boxes, gt_ids, filtered_boxes, filtered_ids, iou_thresh)
    n_gt = len({int(x) for frame in gt_ids for x in frame})
    n_pred = len({int(x) for frame in filtered_ids for x in frame})
    switches = sum(e["event"] == "ID_switch" for e in events)
    summary = dict(counts, IDF1=idf1, ID_switches=switches,
                   fragmentations=sum(e["event"] == "fragmentation" for e in events),
                   n_gt_ids=n_gt, n_pred_ids=n_pred, n_pred_ids_raw=len(raw_ids),
                   count_abs_error=abs(n_pred - n_gt),
                   identity_ratio=n_pred / n_gt if n_gt else float("nan"),
                   switches_per_gt_id=switches / n_gt if n_gt else float("nan"),
                   n_frames=n_frames)
    if pred_scores is not None:
        aps = [frame_ap(g, p, s, threshold)
               for g, p, s in zip(gt_boxes, filtered_boxes, filtered_scores)
               for threshold in np.linspace(0.5, 0.95, 10) if len(g)]
        summary["mAP_frame_mean"] = float(np.mean(aps)) if aps else float("nan")
    return summary, events


Overwriting /kaggle/working/pa2_parte3_modulos/pa2/metrics.py


In [9]:
%%writefile {PACKAGE_DIR}/motion_training.py
"""Eixo 2: fixed GRU, training feedback regime and gradient clipping ablation."""

from __future__ import annotations

import copy
from collections.abc import Sized
from dataclasses import asdict, dataclass
import os
from pathlib import Path
import random
from typing import Literal

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

from pa2.gru_model import GRUSequenceModel

Regime = Literal["teacher_forcing", "scheduled_sampling", "free_running"]
REGIMES: tuple[Regime, ...] = ("teacher_forcing", "scheduled_sampling", "free_running")


@dataclass(frozen=True)
class MotionTrainingConfig:
    regime: Regime = "teacher_forcing"
    seed: int = 0
    epochs: int = 20
    batch_size: int = 256
    hidden_size: int = 32
    learning_rate: float = 1e-3
    clip_norm: float | None = 1.0
    smooth_l1_beta: float = 0.01

    def __post_init__(self):
        if self.regime not in REGIMES:
            raise ValueError(f"Unknown feedback regime: {self.regime}")
        if self.epochs < 2 or self.batch_size < 1 or self.hidden_size < 1:
            raise ValueError("Need at least two epochs, positive batch and hidden sizes")
        if self.learning_rate <= 0 or self.smooth_l1_beta <= 0:
            raise ValueError("Learning rate and Smooth-L1 beta must be positive")
        if self.clip_norm is not None and self.clip_norm <= 0:
            raise ValueError("Clipping norm must be positive or None")


def set_seed(seed: int) -> None:
    os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True)


def teacher_forcing_probability(regime: Regime, epoch: int, epochs: int) -> float:
    if not 0 <= epoch < epochs or epochs < 2:
        raise ValueError("Epoch must be within a schedule of at least two epochs")
    if regime == "teacher_forcing":
        return 1.0
    if regime == "free_running":
        return 0.0
    if regime == "scheduled_sampling":
        return 1.0 - epoch / (epochs - 1)
    raise ValueError(f"Unknown feedback regime: {regime}")


def feedback_box(prediction: torch.Tensor) -> torch.Tensor:
    """Positive sizes for feedback/IoU; supervise the unclamped raw output."""
    return torch.cat((prediction[..., :2], prediction[..., 2:].clamp(min=1e-4)), dim=-1)


def rollout_motion(
    model: GRUSequenceModel, boxes: torch.Tensor, valid: torch.Tensor,
    teacher_probability: float, generator: torch.Generator | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Predict t+1 from t, with one hidden state per identity.

    Inputs: normalized xywh [B,T,4], mask [B,T]. First available GT seeds each
    trajectory, even in free-running. Missing observations use the previous
    prediction, never zero placeholders. A seeded state and valid TARGET suffice
    for supervision, including reappearance after gaps. The graph stays attached
    within the BPTT window; independent dataset windows reset hidden state.
    """
    if boxes.ndim != 3 or boxes.shape[-1] != 4 or boxes.shape[1] < 2:
        raise ValueError("Boxes must have shape [B,T,4] with T >= 2")
    if valid.shape != boxes.shape[:2] or not 0 <= teacher_probability <= 1:
        raise ValueError("Invalid mask shape or teacher-forcing probability")
    if model.bidirectional or not model.batch_first:
        raise ValueError("Motion rollout requires a causal, batch-first GRU")
    batch_size, length = boxes.shape[:2]
    active = torch.zeros(batch_size, dtype=torch.bool, device=boxes.device)
    feedback = torch.zeros_like(boxes[:, 0])
    hidden = None
    predictions, target_masks = [], []
    for t in range(length - 1):
        observed = valid[:, t] > 0
        first = observed & ~active
        if teacher_probability == 1:
            choose_observation = observed
        elif teacher_probability == 0:
            choose_observation = first
        else:
            draw = torch.rand(batch_size, generator=generator).to(boxes.device)
            choose_observation = observed & ((draw < teacher_probability) | first)
        active = active | observed
        inputs = torch.where(choose_observation[:, None], boxes[:, t], feedback)
        inputs = torch.where(active[:, None], inputs, torch.zeros_like(inputs))
        output, next_hidden = model(inputs[:, None], hidden)
        hidden = next_hidden * active[None, :, None]
        prediction = output[:, 0]
        predictions.append(prediction)
        target_masks.append(active & (valid[:, t + 1] > 0))
        feedback = feedback_box(prediction)
    return torch.stack(predictions, dim=1), torch.stack(target_masks, dim=1)


def masked_motion_loss(predictions, targets, mask, beta=0.01) -> torch.Tensor:
    errors = nn.functional.smooth_l1_loss(predictions, targets, beta=beta, reduction="none")
    return torch.where(mask[..., None], errors, torch.zeros_like(errors)).sum() / (mask.sum() * 4).clamp(min=1)


@torch.no_grad()
def evaluate_motion_loss(model, loader, teacher_probability=0.0, beta=0.01) -> float:
    model.eval()
    device = next(model.parameters()).device
    total, coordinates = 0.0, 0
    for boxes, valid in loader:
        boxes, valid = boxes.to(device), valid.to(device)
        predictions, mask = rollout_motion(model, boxes, valid, teacher_probability)
        count = int(mask.sum().item()) * 4
        if count:
            total += masked_motion_loss(predictions, boxes[:, 1:], mask, beta).item() * count
            coordinates += count
    if not coordinates:
        raise ValueError("Validation has no supervised next-box targets")
    return total / coordinates


def _gradient_norm(model: nn.Module) -> torch.Tensor:
    norms = [parameter.grad.detach().norm(2) for parameter in model.parameters()
             if parameter.grad is not None]
    return torch.stack(norms).norm(2)


def fit_motion_model(
    train_data: Dataset, validation_data: Dataset, config: MotionTrainingConfig,
    device: str | torch.device = "cpu",
) -> tuple[GRUSequenceModel, dict, list[dict]]:
    """One seed, best checkpoint selected by validation FREE-RUN loss.

    Record numerical failure and end the run; do not call its earlier checkpoint
    a successful ablation. Separate RNGs pair initialization/minibatch order
    across regimes. Gradient norms are measured BEFORE clipping.
    """
    if not isinstance(train_data, Sized) or not isinstance(validation_data, Sized):
        raise ValueError("Training and validation require finite, sized datasets")
    if not len(train_data) or not len(validation_data):
        raise ValueError("Training and validation datasets must be nonempty")
    set_seed(config.seed)
    shuffle_rng = torch.Generator().manual_seed(config.seed)
    feedback_rng = torch.Generator().manual_seed(config.seed + 100_000)
    train_loader = DataLoader(train_data, batch_size=config.batch_size, shuffle=True, generator=shuffle_rng)
    validation_loader = DataLoader(validation_data, batch_size=config.batch_size, shuffle=False)
    model = GRUSequenceModel(4, config.hidden_size, 4).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.learning_rate)
    best_state, best_loss, best_epoch = None, float("inf"), None
    history = []
    status = "completed"
    for epoch in range(config.epochs):
        probability = teacher_forcing_probability(config.regime, epoch, config.epochs)
        model.train()
        loss_sum, coordinates, grad_norms, clipped_steps = 0.0, 0, [], 0
        for boxes, valid in train_loader:
            boxes, valid = boxes.to(device), valid.to(device)
            optimizer.zero_grad(set_to_none=True)
            predictions, mask = rollout_motion(model, boxes, valid, probability, feedback_rng)
            count = int(mask.sum().item()) * 4
            if not count:
                continue
            loss = masked_motion_loss(predictions, boxes[:, 1:], mask, config.smooth_l1_beta)
            if not torch.isfinite(loss):
                status = "nonfinite_loss"
                break
            loss.backward()
            norm = _gradient_norm(model)
            if not torch.isfinite(norm):
                status = "nonfinite_gradient"
                break
            grad_norms.append(norm.item())
            if config.clip_norm is not None:
                clipped_steps += int(norm.item() > config.clip_norm)
                nn.utils.clip_grad_norm_(model.parameters(), config.clip_norm, error_if_nonfinite=True)
            optimizer.step()
            if any(not torch.isfinite(parameter).all() for parameter in model.parameters()):
                status = "nonfinite_parameters"
                break
            loss_sum += loss.detach().item() * count
            coordinates += count
        if status == "completed" and not coordinates:
            raise ValueError("Training has no supervised next-box targets")
        val_tf = val_free = float("nan")
        if status == "completed":
            val_tf = evaluate_motion_loss(model, validation_loader, 1.0, config.smooth_l1_beta)
            val_free = evaluate_motion_loss(model, validation_loader, 0.0, config.smooth_l1_beta)
            if not np.isfinite(val_tf) or not np.isfinite(val_free):
                status = "nonfinite_validation"
            elif val_free < best_loss:
                best_loss, best_epoch = val_free, epoch + 1
                best_state = copy.deepcopy(model.state_dict())
        history.append({
            "epoch": epoch + 1, "teacher_probability": probability,
            "train_loss": loss_sum / coordinates if coordinates else float("nan"),
            "validation_teacher_loss": val_tf, "validation_free_loss": val_free,
            "gradient_norm_mean": float(np.mean(grad_norms)) if grad_norms else float("nan"),
            "gradient_norm_max": max(grad_norms, default=float("nan")),
            "optimizer_steps": len(grad_norms), "clipped_steps": clipped_steps, "status": status,
        })
        if status != "completed":
            break
    if best_state is not None:
        model.load_state_dict(best_state)
    model.eval()
    metadata = {"config": asdict(config), "status": status, "best_epoch": best_epoch,
                "best_validation_free_loss": best_loss,
                "epochs_completed": sum(row["status"] == "completed" for row in history),
                "maximum_gradient_norm": max((row["gradient_norm_max"] for row in history), default=float("nan"))}
    return model, metadata, history


def load_motion_checkpoint(path: str | Path, device="cpu") -> tuple[GRUSequenceModel, dict]:
    """Restore a successful ablation checkpoint for the colleague's next parts."""
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    if checkpoint["status"] != "completed" or checkpoint["best_epoch"] is None:
        raise ValueError("Checkpoint belongs to an unsuccessful training run")
    config = MotionTrainingConfig(**checkpoint["config"])
    model = GRUSequenceModel(4, config.hidden_size, 4).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])
    model.eval()
    metadata = {key: value for key, value in checkpoint.items() if key != "model_state_dict"}
    return model, metadata


Writing /kaggle/working/pa2_parte3_modulos/pa2/motion_training.py


In [10]:
%%writefile {PACKAGE_DIR}/motion_tracking.py
"""Own causal IoU association, with a GRU state per track or t-1 baseline."""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import torch

from pa2.gru_model import GRUSequenceModel
from pa2.metrics import gated_hungarian, iou_matrix
from pa2.motion_training import feedback_box


@dataclass
class TrackedFrame:
    boxes: np.ndarray  # [N,4], MOT xywh; exactly the observed detector boxes
    ids: np.ndarray    # [N]
    scores: np.ndarray # [N]


@dataclass
class _Track:
    identity: int
    next_box: np.ndarray
    hidden: torch.Tensor | None
    last_seen: int
    missed: int = 0


class MotionTracker:
    def __init__(
        self, image_size: tuple[int, int], model: GRUSequenceModel | None = None,
        association_iou: float = 0.3, max_missed: int = 5,
    ):
        width, height = image_size
        if width <= 0 or height <= 0 or max_missed < 0 or not 0 < association_iou <= 1:
            raise ValueError("Invalid resolution, lifetime or IoU gate")
        if model is not None and (model.bidirectional or not model.batch_first):
            raise ValueError("Tracking needs a causal, batch-first model")
        self.model = model
        if model is not None:
            model.eval()
        self.scale = np.asarray([width, height, width, height], dtype=np.float32)
        self.threshold = association_iou
        self.max_missed = max_missed
        self.tracks: list[_Track] = []
        self.next_id = 1
        self.last_frame: int | None = None

    @torch.inference_mode()
    def _advance(self, track: _Track, observation: np.ndarray) -> None:
        if self.model is None:
            track.next_box = observation.copy()
            return
        device = next(self.model.parameters()).device
        features = torch.as_tensor(observation / self.scale, dtype=torch.float32, device=device)
        output, hidden = self.model(features[None, None], track.hidden)
        track.next_box = feedback_box(output[0, 0]).cpu().numpy() * self.scale
        if not np.isfinite(track.next_box).all() or not torch.isfinite(hidden).all():
            raise FloatingPointError("Nonfinite recurrent state or predicted box")
        track.hidden = hidden

    @torch.inference_mode()
    def update(self, frame: int, detections: np.ndarray) -> TrackedFrame:
        detections = np.asarray(detections, dtype=np.float32).reshape(-1, 5)
        if not np.isfinite(detections).all() or np.any(detections[:, 2:4] <= 0):
            raise ValueError("Detections must be finite xywh + score, with positive sizes")
        if self.last_frame is not None and frame != self.last_frame + 1:
            raise ValueError("Pass every consecutive original frame, including empty frames")
        eligible = [track for track in self.tracks
                    if self.model is not None or track.last_seen == frame - 1]
        predicted = np.asarray([track.next_box for track in eligible]).reshape(-1, 4)
        pairs = gated_hungarian(iou_matrix(predicted, detections[:, :4]), self.threshold)
        ids = np.zeros(len(detections), dtype=np.int64)
        matched_ids, matched_detections = set(), set()
        for track_index, detection_index in pairs:
            track = eligible[track_index]
            ids[detection_index] = track.identity
            matched_ids.add(track.identity)
            matched_detections.add(detection_index)
            self._advance(track, detections[detection_index, :4])
            track.last_seen, track.missed = frame, 0
        for track in self.tracks:
            if track.identity not in matched_ids:
                track.missed += 1
                if track.missed <= self.max_missed and self.model is not None:
                    self._advance(track, track.next_box)
        self.tracks = [track for track in self.tracks if track.missed <= self.max_missed]
        for index, row in enumerate(detections):
            if index in matched_detections:
                continue
            track = _Track(self.next_id, row[:4].copy(), None, frame)
            self.next_id += 1
            self._advance(track, row[:4])
            self.tracks.append(track)
            ids[index] = track.identity
        self.last_frame = frame
        return TrackedFrame(detections[:, :4].copy(), ids, detections[:, 4].copy())


def track_sequence(detections: dict[int, np.ndarray], tracker: MotionTracker) -> dict[int, TrackedFrame]:
    """One persistent tracker for the whole sequence, including empty frames."""
    return {frame: tracker.update(frame, detections[frame]) for frame in sorted(detections)}


Writing /kaggle/working/pa2_parte3_modulos/pa2/motion_tracking.py


In [11]:
%%writefile {PACKAGE_DIR}/ablation_data.py
"""Adapters for existing MOT17 datasets and frozen public detection files."""

from __future__ import annotations

import configparser
from dataclasses import dataclass
import hashlib
from pathlib import Path
from typing import cast

import numpy as np
import torch
from PIL import Image
from torch.utils.data import Dataset, Subset

from pa2.datasets import MOT17_SEQUENCE_SPLITS, make_mot17_track_splits
from pa2.mot17_dataset import MOT17FrameDataset, MOT17TrackDataset, _find_gt


@dataclass
class EvaluationSequence:
    name: str
    split: str
    image_size: tuple[int, int]  # width, height
    detections: dict[int, np.ndarray]  # every frame, [N,5] xywh + score
    gt_boxes: list[np.ndarray]
    gt_ids: list[np.ndarray]
    ignored_boxes: list[np.ndarray]
    detection_sha256: str


def _image_size(sequence_dir: Path) -> tuple[int, int]:
    config = configparser.ConfigParser()
    config.read(sequence_dir / "seqinfo.ini")
    info = config["Sequence"]
    if "imWidth" in info and "imHeight" in info:
        return int(info["imWidth"]), int(info["imHeight"])
    # Small test datasets may omit these fields.
    image = next((sequence_dir / info.get("imDir", "img1")).iterdir())
    with Image.open(image) as opened:
        return opened.size


def _check_frames(frames: list[int], directory: Path) -> None:
    if not frames or frames != list(range(frames[0], frames[-1] + 1)):
        raise ValueError(f"{directory.name}: fixed-dt experiments require consecutive image frames")
    info = configparser.ConfigParser()
    info.read(directory / "seqinfo.ini")
    length = info["Sequence"].getint("seqLength", fallback=None)
    if length is not None and frames != list(range(1, length + 1)):
        raise ValueError(f"{directory.name}: expected all {length} frames declared in seqinfo.ini")


class NormalizedTrackDataset(Dataset):
    """Wrap existing windows; preserve [T,4]/[T] shapes, normalize per video."""

    def __init__(self, subset: Subset[MOT17TrackDataset], scales: dict[str, torch.Tensor]):
        self.subset = subset
        self.scales = scales

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, index):
        underlying_index = self.subset.indices[index]
        dataset = cast(MOT17TrackDataset, self.subset.dataset)
        name = dataset.sample_sequences[underlying_index]
        boxes, valid = self.subset[index]
        return boxes / self.scales[name], valid


def make_ablation_track_splits(root="data", T=16, stride=4, detector="FRCNN"):
    """Reuse repository splits; select one detector copy to avoid triple GT."""
    if T < 2:
        raise ValueError("Motion training needs windows of at least two frames")
    root = Path(root)
    subsets = make_mot17_track_splits(root, T=T, stride=stride)
    frame_data = MOT17FrameDataset(root, split="train")
    scales = {}
    for sequence in frame_data.sequences:
        if sequence.name.endswith("-" + detector):
            directory = root / "MOT17" / "train" / sequence.name
            _check_frames(sequence.frame_numbers, directory)
            width, height = _image_size(directory)
            scales[sequence.name] = torch.tensor([width, height, width, height], dtype=torch.float32)
    output = {}
    for split, subset in subsets.items():
        dataset = cast(MOT17TrackDataset, subset.dataset)
        indices = [int(i) for i in subset.indices if dataset.sample_sequences[i] in scales]
        if not indices:
            raise ValueError(f"No {detector} track windows for {split}; check root and T")
        output[split] = NormalizedTrackDataset(Subset(dataset, indices), scales)
    return output


def _read_table(path: Path, minimum_columns: int) -> np.ndarray:
    if not path.is_file():
        raise FileNotFoundError(path)
    table = (np.loadtxt(path, delimiter=",", ndmin=2) if path.stat().st_size
             else np.empty((0, minimum_columns)))
    if table.shape[1] < minimum_columns or not np.isfinite(table).all():
        raise ValueError(f"Invalid MOT table: {path}")
    return table


def load_evaluation_sequences(root="data", detector="FRCNN", score_threshold=None) -> list[EvaluationSequence]:
    root = Path(root)
    dataset = MOT17FrameDataset(root, split="train")
    output = []
    for sequence in dataset.sequences:
        if not sequence.name.endswith("-" + detector):
            continue
        number = sequence.name.split("-")[1]
        split = next((s for s in ("validation", "test") if number in MOT17_SEQUENCE_SPLITS[s]), None)
        if split is None:
            continue
        directory = root / "MOT17" / "train" / sequence.name
        _check_frames(sequence.frame_numbers, directory)
        det_path = directory / "det" / "det.txt"
        gt_path = _find_gt(root, "train", directory)
        if gt_path is None:
            raise FileNotFoundError(f"No GT for {sequence.name}")
        det_table, gt_table = _read_table(det_path, 7), _read_table(gt_path, 9)
        detections, ignored = {}, []
        for frame in sequence.frame_numbers:
            rows = det_table[det_table[:, 0] == frame]
            keep = (rows[:, 4] > 0) & (rows[:, 5] > 0)
            if score_threshold is not None:
                keep &= rows[:, 6] >= score_threshold
            detections[frame] = rows[keep, 2:7].astype(np.float32)
            detections[frame].setflags(write=False)
            gt_rows = gt_table[gt_table[:, 0] == frame]
            ignore = (gt_rows[:, 6] == 0) | np.isin(gt_rows[:, 7], [2, 7, 8, 12])
            ignored.append(gt_rows[ignore, 2:6])
        output.append(EvaluationSequence(
            sequence.name, split, _image_size(directory), detections,
            [sequence.boxes[f] for f in sequence.frame_numbers],
            [sequence.ids[f] for f in sequence.frame_numbers], ignored,
            hashlib.sha256(det_path.read_bytes()).hexdigest(),
        ))
    if {s.split for s in output} != {"validation", "test"}:
        raise ValueError(f"Need {detector} validation (05) and held-out test (13) sequences")
    return output


Writing /kaggle/working/pa2_parte3_modulos/pa2/ablation_data.py


In [14]:
%%writefile {PACKAGE_DIR}/ablation.py
"""Run and report Part 3 Eixo 2 with existing datasets and the fixed GRU."""

from __future__ import annotations

import csv
from dataclasses import asdict, replace
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch.utils.data import Dataset

from pa2.ablation_data import EvaluationSequence
from pa2.metrics import evaluate_trajectories
from pa2.motion_tracking import MotionTracker, TrackedFrame, track_sequence
from pa2.motion_training import MotionTrainingConfig, REGIMES, fit_motion_model

TRACKING_METRICS = (
    "IDF1", "ID_switches", "fragmentations", "count_abs_error",
    "identity_ratio", "switches_per_gt_id", "mAP_frame_mean",
)


def write_csv(path: Path, rows: list[dict], columns=None) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = list(columns) if columns is not None else list(dict.fromkeys(k for row in rows for k in row))
    with path.open("w", newline="") as opened:
        writer = csv.DictWriter(opened, fieldnames=fields)
        writer.writeheader()
        writer.writerows(rows)


def _evaluate(sequence: EvaluationSequence, predictions: dict[int, TrackedFrame], with_ap=False):
    frames = sorted(sequence.detections)
    summary, events = evaluate_trajectories(
        sequence.gt_boxes, sequence.gt_ids,
        [predictions[f].boxes for f in frames], [predictions[f].ids for f in frames],
        pred_scores=[predictions[f].scores for f in frames] if with_ap else None,
        ignored_boxes=sequence.ignored_boxes,
    )
    for event in events:
        event["frame"] = frames[event.pop("frame_index")]
    return summary, events


def _save_tracks(path: Path, predictions: dict[int, TrackedFrame]) -> None:
    rows = [{"frame": f, "id": int(identity), "x": float(box[0]), "y": float(box[1]),
             "w": float(box[2]), "h": float(box[3]), "score": float(score)}
            for f, result in predictions.items()
            for box, identity, score in zip(result.boxes, result.ids, result.scores)]
    write_csv(path, rows, ("frame", "id", "x", "y", "w", "h", "score"))
    # CSV has no row for an empty frame; retain the evaluated timeline explicitly.
    path.with_suffix(".frames.json").write_text(json.dumps(list(predictions)))


def summarize_seed_metrics(rows: list[dict], expected_seeds=(0, 1, 2)) -> list[dict]:
    """Sample std (ddof=1) over seeds, separately for each video/configuration.

    Report failed/missing runs and successful n. Never present an incomplete
    group as a three-seed result. A single successful seed has undefined std.
    """
    groups = {}
    for row in rows:
        key = (row["sequence"], row["split"], row["regime"], row["clipping"])
        groups.setdefault(key, []).append(row)
    output = []
    for (sequence, split, regime, clipping), group in groups.items():
        seeds = [row["seed"] for row in group]
        if len(seeds) != len(set(seeds)) or not set(seeds) <= set(expected_seeds):
            raise ValueError("Unexpected or duplicate seed within a configuration/video")
        successful = [row for row in group if row["status"] == "completed"]
        result = {"sequence": sequence, "split": split, "regime": regime, "clipping": clipping,
                  "n_expected": len(expected_seeds), "n_successful": len(successful),
                  "n_failed": len(group) - len(successful), "n_missing": len(expected_seeds) - len(group)}
        for metric in TRACKING_METRICS:
            values = np.asarray([row[metric] for row in successful], dtype=float)
            values = values[np.isfinite(values)]
            result[f"{metric}_mean"] = float(values.mean()) if len(values) else float("nan")
            result[f"{metric}_std"] = float(values.std(ddof=1)) if len(values) > 1 else float("nan")
            result[f"{metric}_n"] = len(values)
        output.append(result)
    return output


def _plot_metrics(summary: list[dict], baselines: list[dict], output: Path):
    labels = [f"{short}\n{clip}" for short in ("TF", "SS", "FR") for clip in ("clip", "sin clip")]
    configurations = [(regime, clip) for regime in REGIMES for clip in (True, False)]
    for baseline in baselines:
        selected = {(r["regime"], r["clipping"]): r for r in summary if r["sequence"] == baseline["sequence"]}
        fig, axes = plt.subplots(2, 2, figsize=(11, 7))
        for axis, metric in zip(axes.flat, ("IDF1", "ID_switches", "fragmentations", "count_abs_error")):
            means = np.asarray([selected[key][f"{metric}_mean"] for key in configurations])
            deviations = np.asarray([selected[key][f"{metric}_std"] for key in configurations])
            axis.bar(np.arange(6), means, color=["tab:blue", "tab:orange"] * 3)
            finite = np.isfinite(means) & np.isfinite(deviations)
            axis.errorbar(np.arange(6)[finite], means[finite], yerr=deviations[finite],
                          fmt="none", ecolor="black", capsize=3)
            axis.axhline(baseline[metric], color="black", linestyle="--", label="Baseline IoU t−1")
            axis.set_xticks(np.arange(6), labels)
            axis.set_ylabel(metric)
            axis.grid(axis="y", alpha=0.2)
            if metric == "IDF1":
                axis.set_ylim(0, 1)
            axis.legend(fontsize=8)
        counts = ", ".join(str(selected[key]["n_successful"]) for key in configurations)
        fig.suptitle(f'{baseline["sequence"]} ({baseline["split"]}): media ± DE entre seeds\n'
                     f"Seeds completadas por barra: {counts} / 3")
        fig.tight_layout()
        fig.savefig(output / f'{baseline["sequence"]}_metrics.png', dpi=160)
        plt.close(fig)


def _plot_training(histories: list[dict], output: Path, clip_norm: float):
    fig, axes = plt.subplots(2, 3, figsize=(14, 7), squeeze=False)
    for column, regime in enumerate(REGIMES):
        for clipping, color in ((True, "tab:blue"), (False, "tab:orange")):
            selected = [h for h in histories if h["regime"] == regime and h["clipping"] == clipping]
            epochs = sorted({int(h["epoch"]) for h in selected})
            for metric, axis, style, label in (
                ("validation_free_loss", axes[0, column], "-", "FR validación"),
                ("validation_teacher_loss", axes[0, column], ":", "TF validación"),
                ("gradient_norm_mean", axes[1, column], "-", "gradiente antes de clip"),
            ):
                means, deviations = [], []
                for epoch in epochs:
                    values = np.asarray([h[metric] for h in selected if h["epoch"] == epoch], dtype=float)
                    values = values[np.isfinite(values)]
                    means.append(values.mean() if len(values) else np.nan)
                    deviations.append(values.std(ddof=1) if len(values) > 1 else np.nan)
                means, deviations = np.asarray(means), np.asarray(deviations)
                axis.plot(epochs, means, style, color=color, label=f'{label}, {"clip" if clipping else "sin clip"}')
                axis.fill_between(epochs, means - deviations, means + deviations, color=color, alpha=0.1)
        axes[0, column].set_title(regime)
        axes[0, column].set_ylabel("Smooth-L1 normalizada")
        axes[1, column].set_yscale("log")
        axes[1, column].axhline(clip_norm, color="gray", linestyle="--", label="Umbral de clip")
        axes[1, column].set_ylabel("Norma global del gradiente (L2)")
        for axis in axes[:, column]:
            axis.set_xlabel("Época")
            axis.grid(alpha=0.2)
            axis.legend(fontsize=7)
    fig.suptitle("Eje 2: error de feedback y estabilidad (media ± DE de seeds disponibles)")
    fig.tight_layout()
    fig.savefig(output / "training_and_gradients.png", dpi=160)
    plt.close(fig)


def run_training_ablation(
    train_data: Dataset, validation_data: Dataset, sequences: list[EvaluationSequence],
    output_dir: str | Path, base_config: MotionTrainingConfig = MotionTrainingConfig(),
    seeds: tuple[int, int, int] = (0, 1, 2), clip_norm=1.0,
    association_iou=0.3, max_missed=5, device: str | torch.device = "cpu",
    experiment_metadata: dict | None = None,
) -> tuple[list[dict], list[dict]]:
    """3 regimes × clipping on/off × 3 paired seeds = 18 independent trainings.

    Test trajectories are evaluated only after training/checkpoint selection.
    Validation/test labels never enter the tracker. All configurations receive
    the same immutable detections, frames, matching gate and lifetime.
    """
    if len(seeds) != 3 or len(set(seeds)) != 3:
        raise ValueError("Part 3 requires exactly three distinct seeds")
    if not sequences or clip_norm <= 0:
        raise ValueError("Need evaluation sequences and a positive clipping norm")
    output = Path(output_dir)
    output.mkdir(parents=True, exist_ok=True)
    manifest = {
        "axis": 2, "seeds": seeds, "base_training": asdict(base_config),
        "regimes": REGIMES, "clip_norm_on": clip_norm, "n_trainings": 18,
        "association_iou": association_iou, "evaluation_iou": 0.5, "max_missed": max_missed,
        "box_format": "MOT one-based xywh; normalized by each video's [W,H,W,H]",
        "checkpoint_selection": "minimum validation free-running Smooth-L1",
        "sequences": [{"name": s.name, "split": s.split, "frames": sorted(s.detections),
                       "detection_sha256": s.detection_sha256} for s in sequences],
        "torch": torch.__version__, "numpy": np.__version__, "device": str(device),
        "experiment": experiment_metadata or {},
    }
    (output / "experiment.json").write_text(json.dumps(manifest, indent=2))
    baselines = []
    for sequence in sequences:
        predictions = track_sequence(sequence.detections, MotionTracker(
            sequence.image_size, association_iou=association_iou, max_missed=max_missed))
        summary, events = _evaluate(sequence, predictions, with_ap=True)
        baselines.append(dict(sequence=sequence.name, split=sequence.split, **summary))
        _save_tracks(output / "baseline" / f"{sequence.name}_tracks.csv", predictions)
        write_csv(output / "baseline" / f"{sequence.name}_events.csv", events,
                  ("frame", "event", "gt_id", "old_pred_id", "new_pred_id"))
    write_csv(output / "baseline.csv", baselines)
    baseline_by_name = {row["sequence"]: row for row in baselines}
    rows, histories, training_runs = [], [], []
    for regime in REGIMES:
        for clipping in (True, False):
            for seed in seeds:
                config = replace(base_config, regime=regime, seed=seed,
                                 clip_norm=clip_norm if clipping else None)
                run_name = f'{regime}_clip_{"on" if clipping else "off"}_seed_{seed}'
                print(f"Training {run_name}", flush=True)
                run_dir = output / run_name
                run_dir.mkdir(parents=True, exist_ok=True)
                model, metadata, history = fit_motion_model(train_data, validation_data, config, device)
                histories.extend(dict(regime=regime, clipping=clipping, seed=seed, **h) for h in history)
                write_csv(run_dir / "history.csv", history)
                # State dict and config suffice to reconstruct the existing GRU.
                torch.save({"model_state_dict": model.state_dict(), **metadata}, run_dir / "checkpoint.pt")
                evaluation_failed = False
                for sequence in sequences:
                    result = {"sequence": sequence.name, "split": sequence.split, "regime": regime,
                              "clipping": clipping, "seed": seed, "status": metadata["status"],
                              "best_epoch": metadata["best_epoch"]}
                    if metadata["status"] == "completed":
                        try:
                            predictions = track_sequence(sequence.detections, MotionTracker(
                                sequence.image_size, model, association_iou, max_missed))
                            summary, events = _evaluate(sequence, predictions)
                            # Output boxes/scores are identical for every tracker; detector AP is invariant.
                            summary["mAP_frame_mean"] = baseline_by_name[sequence.name]["mAP_frame_mean"]
                            result.update(summary)
                            _save_tracks(run_dir / f"{sequence.name}_tracks.csv", predictions)
                            write_csv(run_dir / f"{sequence.name}_events.csv", events,
                                      ("frame", "event", "gt_id", "old_pred_id", "new_pred_id"))
                        except FloatingPointError:
                            result["status"] = "nonfinite_inference"
                            evaluation_failed = True
                    if result["status"] != "completed":
                        result.update({metric: float("nan") for metric in TRACKING_METRICS})
                    rows.append(result)
                metadata["evaluation_failed"] = evaluation_failed
                (run_dir / "metadata.json").write_text(json.dumps(metadata, indent=2))
                training_runs.append(dict(regime=regime, clipping=clipping, seed=seed,
                                          **{k: v for k, v in metadata.items() if k != "config"}))
                # Persist after every seed so an interrupted Kaggle session retains finished work.
                write_csv(output / "run_results.csv", rows)
                write_csv(output / "training_runs.csv", training_runs)
                print(f'{run_name}: {metadata["status"]}, best epoch {metadata["best_epoch"]}', flush=True)
    summary_rows = summarize_seed_metrics(rows, seeds)
    write_csv(output / "seed_summary.csv", summary_rows)
    _plot_metrics(summary_rows, baselines, output)
    _plot_training(histories, output, clip_norm)
    return rows, summary_rows


Overwriting /kaggle/working/pa2_parte3_modulos/pa2/ablation.py


In [15]:
# Priorize o pacote gravado pelo próprio notebook, inclusive ao reexecutar células.
sys.path.insert(0, str(SOURCE_ROOT))
for module_name in list(sys.modules):
    if module_name == "pa2" or module_name.startswith("pa2."):
        del sys.modules[module_name]
importlib.invalidate_caches()

import pa2
from dataclasses import replace
from torch.utils.data import Subset
from pa2.ablation import run_training_ablation, TRACKING_METRICS
from pa2.ablation_data import make_ablation_track_splits, load_evaluation_sequences
from pa2.datasets import MOT17_SEQUENCE_SPLITS
from pa2.motion_training import MotionTrainingConfig, load_motion_checkpoint
from pa2.motion_tracking import MotionTracker, track_sequence

assert Path(pa2.__file__).parent.resolve() == PACKAGE_DIR.resolve()
print("Pacote carregado:", pa2.__file__)


Pacote carregado: /kaggle/working/pa2_parte3_modulos/pa2/__init__.py


In [21]:
import configparser
import re

def resolve_mot17_root(configured=None, search_roots=None):
    """Find FRCNN videos in canonical or flat layouts; validate files first."""
    required = set(number for group in MOT17_SEQUENCE_SPLITS.values() for number in group)
    roots = []
    if configured is not None:
        chosen = Path(configured).expanduser()
        if chosen.is_dir():
            roots.append(chosen.parent if (chosen / "seqinfo.ini").is_file() else chosen)
        else:
            print("DATA_ROOT configurado não existe:", chosen)
    if search_roots is None:
        search_roots = (
            Path("/kaggle/input"), BASE_DIR / "mot17", BASE_DIR / "MOT17", BASE_DIR / "data",
        )
    roots += [Path(root) for root in search_roots if Path(root).is_dir()]
    roots = list(dict.fromkeys(roots))
    print("Pastas pesquisadas:", ", ".join(map(str, roots)) or "(nenhuma disponível)")
    input_root = Path("/kaggle/input")
    if input_root.is_dir():
        print("Entradas em /kaggle/input:")
        for entry in sorted(input_root.iterdir())[:15]:
            print(" -", entry)

    infos = []
    for root in roots:
        found = sorted(root.rglob("seqinfo.ini"))
        print(f"{root}: {len(found)} arquivos seqinfo.ini")
        infos.extend(found)
        for ini in found[:8]:
            print("  ", ini)
    infos = list(dict.fromkeys(ini.resolve() for ini in infos))

    groups = {}
    pattern = re.compile(r"MOT17-(\d{2})-" + re.escape(PUBLIC_DETECTOR) + r"$")
    for ini in infos:
        match = pattern.fullmatch(ini.parent.name)
        if match and match.group(1) in required:
            groups.setdefault(ini.parent.parent, {})[match.group(1)] = ini.parent

    if not groups:
        detector_files = list(dict.fromkeys(p for root in roots for p in root.rglob("det.txt")))
        gt_files = list(dict.fromkeys(p for root in roots for p in root.rglob("gt.txt")))
        print("det.txt encontrados:", len(detector_files), "| gt.txt encontrados:", len(gt_files))
        for candidate in (detector_files + gt_files)[:10]:
            print("  ", candidate)
        if infos:
            print("Pastas de sequência encontradas:", ", ".join(sorted({p.parent.name for p in infos})))
        raise FileNotFoundError(
            f"Não encontrei sequências MOT17-{PUBLIC_DETECTOR} com seqinfo.ini. "
            "No Kaggle, use Add Input / Add Data para anexar o dataset real com "
            "img1, seqinfo.ini, gt/gt.txt e det/det.txt. Se há apenas HTML, GIF ou "
            "notebooks, esses arquivos são resultados e não permitem treinar. "
            "Se o MOT17 está em um ZIP, extraia-o em /kaggle/working/mot17. "
            "Se as imagens existem sem seqinfo.ini, é preciso o metadata original."
        )

    complete = [parent for parent, videos in groups.items() if required <= set(videos)]
    if len(complete) > 1 and configured is not None:
        selected = Path(configured).resolve()
        preferred = [parent for parent in complete if parent == selected or selected in parent.parents]
        if len(preferred) == 1:
            complete = preferred
    if len(complete) != 1:
        for parent, videos in sorted(groups.items()):
            print("Grupo:", parent, "| presentes:", sorted(videos),
                  "| faltam:", sorted(required - set(videos)))
        if not complete:
            raise FileNotFoundError(
                f"Preciso das sequências {sorted(required)} do detector {PUBLIC_DETECTOR} "
                "na mesma pasta para os splits fixos de treino/validação/teste. "
                "Confira o dataset anexado e DATA_ROOT."
            )
        raise ValueError("Mais de um dataset completo foi encontrado. Ajuste DATA_ROOT para um dos grupos acima.")

    parent = complete[0]
    videos = groups[parent]
    sources, gt_paths, missing = {}, {}, []
    for number in sorted(required):
        source = videos[number]
        sources[source.name] = source
        config = configparser.ConfigParser()
        config.read(source / "seqinfo.ini")
        if not config.has_section("Sequence"):
            missing.append(f"{source.name}: seção [Sequence] em seqinfo.ini")
            continue
        images = source / config["Sequence"].get("imDir", "img1")
        extension = config["Sequence"].get("imExt", ".jpg")
        extension = extension if extension.startswith(".") else "." + extension
        if not images.is_dir() or next(images.glob("*" + extension), None) is None:
            missing.append(f"{source.name}: imagens {images}")
        if not (source / "det" / "det.txt").is_file():
            missing.append(f"{source.name}: det/det.txt")
        candidates = [source / "gt" / "gt.txt"]
        candidates += [ancestor / "MOT17Labels" / "train" / source.name / "gt" / "gt.txt"
                       for ancestor in list(source.parents)[:4]]
        gt = next((p for p in candidates if p.is_file()), None)
        if gt is None:
            # Ground truth can be attached as a separate Kaggle input.
            external = [p for root in roots for p in root.rglob("gt.txt")
                        if p.parent.name == "gt" and p.parent.parent.name == source.name]
            gt = next(iter(external), None)
        if gt is None:
            missing.append(f"{source.name}: gt/gt.txt (ou MOT17Labels/train/{source.name}/gt/gt.txt)")
        else:
            gt_paths[source.name] = gt
    if missing:
        print("Arquivos faltantes:")
        for item in missing:
            print(" -", item)
        raise FileNotFoundError("Dataset incompleto. Adicione os arquivos listados antes de treinar.")

    canonical = parent.name == "train" and parent.parent.name == "MOT17"
    inline_gt = all((source / "gt" / "gt.txt").is_file() for source in sources.values())
    original_labels = all(
        (parent.parent.parent / "MOT17Labels" / "train" / name / "gt" / "gt.txt").is_file()
        or (source / "gt" / "gt.txt").is_file()
        for name, source in sources.items()
    )
    if canonical and (inline_gt or original_labels):
        print("Dataset completo encontrado:", parent.parent.parent)
        return parent.parent.parent

    # Expose the existing files in the repository layout, without copying images.
    view = BASE_DIR / "pa2_mot17_layout"
    for name, source in sources.items():
        destination = view / "MOT17" / "train" / name
        if destination.resolve() == source.resolve():
            continue
        destination.mkdir(parents=True, exist_ok=True)
        links = {child.name: child for child in source.iterdir() if child.name != "gt"}
        links["gt"] = gt_paths[name].parent
        for child_name, original in links.items():
            link = destination / child_name
            if link.is_symlink():
                if link.resolve() == original.resolve():
                    continue
                link.unlink()
            elif link.exists():
                raise FileExistsError(f"Arquivo preexistente no layout de trabalho: {link}")
            link.symlink_to(original.resolve(), target_is_directory=original.is_dir())
    print("Layout de origem:", parent)
    print("Layout compatível criado:", view, "(links; nenhuma imagem copiada)")
    return view

DATA_ROOT = resolve_mot17_root(DATA_ROOT)
print("DATA_ROOT:", DATA_ROOT)
for split, numbers in MOT17_SEQUENCE_SPLITS.items():
    print(split, ":", ", ".join(numbers))


Pastas pesquisadas: /kaggle/input/datasets/ahmedsamir1598/mot17challenge, /kaggle/input
Entradas em /kaggle/input:
 - /kaggle/input/datasets
/kaggle/input/datasets/ahmedsamir1598/mot17challenge: 42 arquivos seqinfo.ini
   /kaggle/input/datasets/ahmedsamir1598/mot17challenge/MOT17/test/MOT17-01-DPM/seqinfo.ini
   /kaggle/input/datasets/ahmedsamir1598/mot17challenge/MOT17/test/MOT17-01-FRCNN/seqinfo.ini
   /kaggle/input/datasets/ahmedsamir1598/mot17challenge/MOT17/test/MOT17-01-SDP/seqinfo.ini
   /kaggle/input/datasets/ahmedsamir1598/mot17challenge/MOT17/test/MOT17-03-DPM/seqinfo.ini
   /kaggle/input/datasets/ahmedsamir1598/mot17challenge/MOT17/test/MOT17-03-FRCNN/seqinfo.ini
   /kaggle/input/datasets/ahmedsamir1598/mot17challenge/MOT17/test/MOT17-03-SDP/seqinfo.ini
   /kaggle/input/datasets/ahmedsamir1598/mot17challenge/MOT17/test/MOT17-06-DPM/seqinfo.ini
   /kaggle/input/datasets/ahmedsamir1598/mot17challenge/MOT17/test/MOT17-06-FRCNN/seqinfo.ini
/kaggle/input: 42 arquivos seqinfo.ini


In [19]:
splits = make_ablation_track_splits(
    DATA_ROOT, T=WINDOW, stride=STRIDE, detector=PUBLIC_DETECTOR
)
sequences = load_evaluation_sequences(DATA_ROOT, detector=PUBLIC_DETECTOR)
train_data, validation_data = splits["train"], splits["validation"]

if SMOKE_TEST:
    train_data = Subset(train_data, range(min(SMOKE_MAX_WINDOWS, len(train_data))))
    validation_data = Subset(validation_data, range(min(SMOKE_MAX_WINDOWS, len(validation_data))))
    shortened = []
    for sequence in sequences:
        frames = sorted(sequence.detections)[:SMOKE_MAX_FRAMES]
        length = len(frames)
        shortened.append(replace(
            sequence, detections={f: sequence.detections[f] for f in frames},
            gt_boxes=sequence.gt_boxes[:length], gt_ids=sequence.gt_ids[:length],
            ignored_boxes=sequence.ignored_boxes[:length],
        ))
    sequences = shortened

print("Janelas de treino:", len(train_data))
print("Janelas de validação:", len(validation_data))
print("Janela BPTT:", WINDOW, "| stride:", STRIDE)
for sequence in sequences:
    gt_id_count = len({int(i) for frame in sequence.gt_ids for i in frame})
    print(sequence.name, "|", sequence.split, "| frames:", len(sequence.detections),
          "| resolução:", sequence.image_size, "| IDs GT:", gt_id_count)
    print("  SHA-256 det.txt:", sequence.detection_sha256)
assert {s.split for s in sequences} == {"validation", "test"}
assert len(SEEDS) == len(set(SEEDS)) == 3


Janelas de treino: 23677
Janelas de validação: 2081
Janela BPTT: 16 | stride: 4
MOT17-05-FRCNN | validation | frames: 837 | resolução: (640, 480) | IDs GT: 133
  SHA-256 det.txt: 6dfc37078b989f9e7f0e2f0492836ef28a2b51ffcf30ded428fd61d53e3aca98
MOT17-13-FRCNN | test | frames: 750 | resolução: (1920, 1080) | IDs GT: 110
  SHA-256 det.txt: d95a3a26b64a2164d6608ed5cc732fe7f63fc5157c884f218e43cd04507b9989


In [ ]:
config = MotionTrainingConfig(
    epochs=2 if SMOKE_TEST else EPOCHS,
    batch_size=BATCH_SIZE,
    hidden_size=HIDDEN_SIZE,
    learning_rate=LEARNING_RATE,
    smooth_l1_beta=SMOOTH_L1_BETA,
)

rows, seed_summary = run_training_ablation(
    train_data, validation_data, sequences, OUTPUT,
    base_config=config, seeds=SEEDS, clip_norm=CLIP_NORM,
    association_iou=ASSOCIATION_IOU, max_missed=MAX_MISSED, device=DEVICE,
    experiment_metadata={
        "data_root": str(DATA_ROOT), "detector": PUBLIC_DETECTOR,
        "window": WINDOW, "stride": STRIDE, "smoke_test": SMOKE_TEST,
        "purpose": "verificação de execução" if SMOKE_TEST else "Parte 3 — Eixo 2",
    },
)
print("Execução concluída:", OUTPUT)


Training teacher_forcing_clip_on_seed_0
teacher_forcing_clip_on_seed_0: completed, best epoch 20
Training teacher_forcing_clip_on_seed_1
teacher_forcing_clip_on_seed_1: completed, best epoch 12
Training teacher_forcing_clip_on_seed_2
teacher_forcing_clip_on_seed_2: completed, best epoch 11
Training teacher_forcing_clip_off_seed_0
teacher_forcing_clip_off_seed_0: completed, best epoch 20
Training teacher_forcing_clip_off_seed_1
teacher_forcing_clip_off_seed_1: completed, best epoch 12
Training teacher_forcing_clip_off_seed_2
teacher_forcing_clip_off_seed_2: completed, best epoch 11
Training scheduled_sampling_clip_on_seed_0
scheduled_sampling_clip_on_seed_0: completed, best epoch 20
Training scheduled_sampling_clip_on_seed_1
scheduled_sampling_clip_on_seed_1: completed, best epoch 20
Training scheduled_sampling_clip_on_seed_2
scheduled_sampling_clip_on_seed_2: completed, best epoch 20
Training scheduled_sampling_clip_off_seed_0
scheduled_sampling_clip_off_seed_0: completed, best epoch 1

In [ ]:
def read_csv_rows(path):
    with Path(path).open(newline="") as opened:
        return list(csv.DictReader(opened))

def show_table(table_rows, columns):
    header = "".join("<th>" + html.escape(str(c)) + "</th>" for c in columns)
    body = ""
    for row in table_rows:
        body += "<tr>" + "".join("<td>" + html.escape(str(row.get(c, ""))) + "</td>"
                                for c in columns) + "</tr>"
    display(HTML("<div style='overflow-x:auto'><table><thead><tr>" + header +
                 "</tr></thead><tbody>" + body + "</tbody></table></div>"))

def mean_std_text(row, metric):
    mean, std = float(row[metric + "_mean"]), float(row[metric + "_std"])
    if not np.isfinite(mean):
        return "sem resultado válido"
    return f"{mean:.4f} ± {std:.4f}" if np.isfinite(std) else f"{mean:.4f} (DE indefinido)"

saved_summary = read_csv_rows(OUTPUT / "seed_summary.csv")
saved_baseline = read_csv_rows(OUTPUT / "baseline.csv")
saved_runs = read_csv_rows(OUTPUT / "training_runs.csv")

comparison_rows = []
for row in saved_summary:
    display_row = {key: row[key] for key in ("sequence", "split", "regime")}
    display_row["clipping"] = "ligado" if row["clipping"] == "True" else "desligado"
    display_row["seeds válidas"] = row["n_successful"] + "/" + row["n_expected"]
    for metric in ("IDF1", "ID_switches", "fragmentations", "count_abs_error",
                   "identity_ratio", "switches_per_gt_id", "mAP_frame_mean"):
        display_row[metric] = mean_std_text(row, metric)
    comparison_rows.append(display_row)

if SMOKE_TEST:
    display(HTML("<b>SMOKE TEST: tabelas apenas para verificar o pipeline.</b>"))
show_table(comparison_rows, (
    "sequence", "split", "regime", "clipping", "seeds válidas", "IDF1", "ID_switches",
    "fragmentations", "count_abs_error", "identity_ratio", "switches_per_gt_id", "mAP_frame_mean",
))
display(HTML("<b>Baseline IoU estrita t−1, nas mesmas sequências</b>"))
show_table(saved_baseline, (
    "sequence", "split", "IDF1", "ID_switches", "fragmentations", "count_abs_error",
    "identity_ratio", "mAP_frame_mean",
))


In [ ]:
for sequence in sequences:
    print(sequence.name, "|", sequence.split)
    display(Image(filename=str(OUTPUT / f"{sequence.name}_metrics.png")))


In [ ]:
show_table(saved_runs, (
    "regime", "clipping", "seed", "status", "best_epoch", "epochs_completed",
    "best_validation_free_loss", "maximum_gradient_norm", "evaluation_failed",
))
gradient_rows = []
feedback_rows = []
for run in saved_runs:
    clip_on = run["clipping"] == "True"
    run_name = (run["regime"] + "_clip_" + ("on" if clip_on else "off") +
                "_seed_" + run["seed"])
    history = read_csv_rows(OUTPUT / run_name / "history.csv")
    steps = sum(int(h["optimizer_steps"]) for h in history)
    clipped = sum(int(h["clipped_steps"]) for h in history)
    gradient_rows.append({
        "regime": run["regime"], "clipping": run["clipping"], "seed": run["seed"],
        "status": run["status"], "norma máxima antes de clip": run["maximum_gradient_norm"],
        "passos": steps, "passos cortados": clipped,
        "fração cortada": f"{clipped / steps:.3f}" if steps else "sem passos válidos",
    })
    if run["best_epoch"] not in ("", "None"):
        best = next(h for h in history if h["epoch"] == run["best_epoch"])
        teacher = float(best["validation_teacher_loss"])
        free = float(best["validation_free_loss"])
        feedback_rows.append({
            "regime": run["regime"], "clipping": run["clipping"], "seed": run["seed"],
            "melhor época": run["best_epoch"], "loss TF": f"{teacher:.6f}",
            "loss FR": f"{free:.6f}", "FR − TF": f"{free - teacher:.6f}",
        })
show_table(gradient_rows, (
    "regime", "clipping", "seed", "status", "norma máxima antes de clip",
    "passos", "passos cortados", "fração cortada",
))
show_table(feedback_rows, (
    "regime", "clipping", "seed", "melhor época", "loss TF", "loss FR", "FR − TF",
))
display(Image(filename=str(OUTPUT / "training_and_gradients.png")))


In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED

module_zip = OUTPUT / "funcoes_parte3.zip"
with ZipFile(module_zip, "w", compression=ZIP_DEFLATED) as archive:
    for path in sorted(PACKAGE_DIR.glob("*.py")):
        archive.write(path, path.relative_to(SOURCE_ROOT))
display(FileLink(str(module_zip)))
display(FileLink(str(OUTPUT / "seed_summary.csv")))

results_zip = BASE_DIR / ("parte3_smoke_resultados.zip" if SMOKE_TEST else "parte3_resultados.zip")
with ZipFile(results_zip, "w", compression=ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT.rglob("*")):
        if path.is_file():
            archive.write(path, path.relative_to(OUTPUT))
display(FileLink(str(results_zip)))
print("Resultados exportados:", results_zip)
